# Refinitiv Workspace News Smoke Test

This notebook is a starter workflow for pulling news programmatically from Refinitiv/LSEG Workspace.

The original paper uses Thomson Reuters News Analytics (TRNA). This notebook starts with the Workspace news APIs so we can verify access, inspect returned fields, and export reproducible raw samples before building the sentiment pipeline.

It verifies that we can:

- Import the LSEG Data Library for Python, with a fallback for older `refinitiv.data` installs.
- Open a local Workspace desktop session.
- Pull news headlines for a RIC/query and date range.
- Pull full story text for a returned `storyId`.
- Inspect whether returned fields include TRNA-style sentiment metadata.
- Export raw headline and story samples under `data/raw/news/refinitiv/`.

Keep Refinitiv/LSEG Workspace open and signed in before running the connection cell.

## Optional Dependency

Install the optional LSEG/Refinitiv client in the project environment:

```bash
conda activate sentiment-ltr-paper
pip install -r requirements-refinitiv.txt
```

The current package is `lseg-data`. Older Workspace/Eikon examples may use `refinitiv-data`; the import cell below tries both so we can see what is available locally.

In [ ]:
from pathlib import Path
import os
import sys
from datetime import datetime, timezone

import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

load_dotenv(PROJECT_ROOT / ".env")

OUTPUT_DIR = PROJECT_ROOT / "data" / "raw" / "news" / "refinitiv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

## Import The Data Library

`lseg.data` is the current LSEG Data Library package. If your local Workspace setup still uses the legacy Refinitiv package, install `refinitiv-data` and the fallback import should identify it.

In [ ]:
try:
    import lseg.data as rd
    DATA_LIBRARY = "lseg.data"
except ImportError:
    import refinitiv.data as rd
    DATA_LIBRARY = "refinitiv.data"

print(f"Using {DATA_LIBRARY}")
print(f"Package version: {getattr(rd, '__version__', 'unknown')}")

## Open A Workspace Session

For a desktop session, Workspace must be running locally and signed in. Paste your Workspace App Key into the ignored local config file:

```bash
cp lseg-data.config.example.json lseg-data.config.json
```

With `lseg-data` 2.x, the App Key in `lseg-data.config.json` is not picked up reliably by `open_session(config_name=...)`. The notebook therefore reads the key from that file and applies it with `get_config().set_param(...)` before opening the session.

You can also set `LSEG_APP_KEY` in `.env` instead of editing the JSON file.

In [ ]:
from sentiment_ltr.data.refinitiv_session import open_workspace_session

session = open_workspace_session(PROJECT_ROOT, rd)
session

In [ ]:
smoke = rd.get_data(
    universe=["AAPL.O", "MSFT.O"],
    fields=["BID", "ASK", "TR.Revenue"],
)
smoke

In [ ]:
The session stays open for the news queries below. Close it in the final cell when you are done.

## Pull News Headlines

The query language is the same one used by Workspace News Monitor. Start small, then scale once field coverage and limits are clear.

Examples:

- `R:AAPL.O AND Language:LEN`
- `R:MSFT.O AND Language:LEN`
- `Topic:STX AND Language:LEN`

For the paper pipeline, we will eventually generate these queries from the stock universe's RIC/ticker mapping.

In [ ]:
NEWS_QUERY = "R:AAPL.O AND Language:LEN"
START = "2024-01-01"
END = "2024-01-31"
COUNT = 100

headlines = rd.news.get_headlines(NEWS_QUERY, start=START, end=END, count=COUNT)
print(f"Rows: {len(headlines):,}")
print(f"Columns: {list(headlines.columns)}")
headlines.head()

## Inspect Candidate TRNA/Sentiment Fields

The paper expects article-level fields like `pos`, `neg`, `obj`, and `relevance`. Workspace headline results may only include standard news metadata unless your entitlement includes machine-readable analytics fields. This cell checks returned column names for likely sentiment or relevance fields.

In [ ]:
sentiment_keywords = ["sent", "tone", "pos", "neg", "obj", "relev", "score", "prob"]
candidate_sentiment_cols = [
    column for column in headlines.columns
    if any(keyword in str(column).lower() for keyword in sentiment_keywords)
]

if candidate_sentiment_cols:
    display(headlines[candidate_sentiment_cols].head())
else:
    print("No obvious TRNA-style sentiment/relevance fields found in the headline response.")
    print("Next step: confirm whether your entitlement exposes News Analytics fields through Workspace or whether we need a separate analytics endpoint/feed.")

## Pull One Full Story

If the headline response includes `storyId`, pull the first story. LSEG story responses can be HTML; downstream text processing should preserve the raw story and create a cleaned text column separately.

In [ ]:
story_id_column = next((column for column in headlines.columns if str(column).lower() == "storyid"), None)

if story_id_column is None or headlines.empty:
    story_id = None
    story_text = None
    print("No storyId column/rows available for story retrieval.")
else:
    story_id = headlines[story_id_column].dropna().iloc[0]
    story_text = rd.news.get_story(story_id)
    print(f"Story ID: {story_id}")
    print(str(story_text)[:2000])

## Normalize A Raw Sample

This creates a thin raw sample table. We intentionally keep vendor-native fields and add only lightweight metadata so we can preserve auditability before deciding how to map fields into the paper's TRNA schema.

In [ ]:
raw_sample = headlines.copy()
raw_sample.insert(0, "query", NEWS_QUERY)
raw_sample.insert(1, "pulled_at_utc", datetime.now(timezone.utc).isoformat())
raw_sample.insert(2, "data_library", DATA_LIBRARY)

if story_id is not None:
    raw_sample.loc[raw_sample[story_id_column] == story_id, "sample_story_text"] = str(story_text)

raw_sample.head()

## Export Raw Samples

The output path is ignored by Git. Commit only schemas/manifests or aggregated validation artifacts, not licensed raw news data.

In [ ]:
safe_query = "".join(char.lower() if char.isalnum() else "_" for char in NEWS_QUERY).strip("_")[:80]
output_path = OUTPUT_DIR / f"headlines_{safe_query}_{START}_{END}.csv"
raw_sample.to_csv(output_path, index=False)

manifest = pd.DataFrame([
    {
        "output_file": str(output_path.relative_to(PROJECT_ROOT)),
        "query": NEWS_QUERY,
        "start": START,
        "end": END,
        "count_requested": COUNT,
        "rows_returned": len(raw_sample),
        "columns": ",".join(map(str, raw_sample.columns)),
        "data_library": DATA_LIBRARY,
        "pulled_at_utc": datetime.now(timezone.utc).isoformat(),
    }
])
manifest_path = OUTPUT_DIR / f"headlines_{safe_query}_{START}_{END}_manifest.csv"
manifest.to_csv(manifest_path, index=False)

print(output_path)
print(manifest_path)

## Close Session

Close the Workspace session when the smoke test is complete.

In [ ]:
rd.close_session()

## Follow-Up Checks

After the smoke test works locally:

- Confirm which query pattern returns company-specific historical news for each stock universe member.
- Confirm whether your entitlement returns TRNA/News Analytics fields or only headline/story metadata.
- Identify the stable company identifier to join against market data: RIC, ticker, PermID, ISIN, or another mapping field.
- Test pagination or batching limits before requesting the full 2003-2014 panel.
- Build a manifest per batch with query, date range, row counts, columns, failures, and pull timestamp.
- Keep raw licensed news data in `data/raw/news/refinitiv/` and out of Git.